# Pillar 4 scoring - market quality and durability

This notebook reads two Parquet datasets: the metro-year panel and NAICS sector panel and calculates the Pillar 4 scoring framework. 

## How to run

1. Confrim and update their paths below `../data/processed/msa_year.parquet` and `../data/processed/msa_naics_year.parquet`.
2. Install the dependencies if needed: `pip install pandas numpy pyarrow` and run in Jupyter, VS Code, or another Python notebook environment.
3. Run the cells in order. Change `FOCUS_CBSA` to inspect a market; Charlotte is `16740`.

## What the supplied dataset supports

The supplied panel covers **2015–2024**. The default `dataset_2024` mode applies the reference scoring method to the latest comparable periods in this panel: nine GDP growth observations for volatility, 2019–2024 population growth, and 2024 net migration. This notebook periods require 2014 GDP and 2025 demographics; select `reference_periods` to see those strict definitions without substituting years. Missing years remain missing.

The NAICS dataset supplies 2023 County Business Patterns (CBP) employment for 19 classified sectors. The notebook groups them into ten categories and calculates an industry-employment HHI only where all 19 source sectors are reported. This is a 2023 CBP measure based on the March pay period, not the reference workbook's 2024 QCEW annual-average employment. Missing sector rows are not assumed to be zero, and reported sector totals can still omit small county-sector observations. The all-industry QCEW total and bank-deposit HHI are not substitutes.

The default mode produces an **adapted eight-metric score** for metros with all eight measures and a separately labeled available-data score for partial records. The reference-period mode leaves the industry HHI blank because its original 2024 QCEW sector inputs are not supplied. Both modes disclose their data coverage.

The model is a proposed Pillar 4 measure, not a complete market-attractiveness or lending-opportunity score. The original reference definitions and this notebook's adapted periods are different; their numerical scores and ranks should not be compared as like-for-like results.

## 1. Configuration

`dataset_2024` uses fixed periods, so every market is evaluated over the same years; industry HHI uses the latest available CBP sector year, 2023. `reference_periods` keeps the workbook's dates: GDP volatility 2015–2024, population CAGR 2020–2025, migration rate 2025, and QCEW-based industry HHI 2024. Those missing reference inputs remain blank; unemployment still comes from the county-derived panel and will not exactly reproduce the reference source. Edit the weights if desired; all eight must be positive and sum to 1.

In [2]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display

DATA_PATH = Path("../data/processed/msa_year.parquet")
INDUSTRY_DATA_PATH = Path("../data/processed/msa_naics_year.parquet")
INDUSTRY_YEAR = 2023
DEFINITION_MODE = "dataset_2024"   # "dataset_2024" or "reference_periods"
FOCUS_CBSA = "16740"               # Charlotte
SENSITIVITY = 0.25                 # one metric at a time: +/− 25% of its weight
EXPORT_CSV = False
OUTPUT_PATH = Path("pillar4_scores.csv")

METRIC_LABELS = {
    "gdp_growth": "GDP growth",
    "gdp_volatility": "GDP volatility",
    "unemployment_level": "Unemployment level",
    "unemployment_volatility": "Unemployment volatility",
    "industry_hhi": "Industry HHI",
    "establishment_growth": "Establishment growth",
    "population_growth": "Population growth",
    "net_migration_rate": "Net migration /1,000",
}
METRICS = list(METRIC_LABELS)
INTERIM_METRICS = [m for m in METRICS if m != "industry_hhi"]
INDUSTRY_GROUPS = {
    "Natural resources and mining": ["11", "21"],
    "Construction": ["23"],
    "Manufacturing": ["31"],  # 31 represents NAICS 31–33
    "Trade, transportation, and utilities": ["22", "42", "44", "48"],
    "Information": ["51"],
    "Financial activities": ["52", "53"],
    "Professional and business services": ["54", "55", "56"],
    "Education and health services": ["61", "62"],
    "Leisure and hospitality": ["71", "72"],
    "Other services": ["81"],
}
CLASSIFIED_SECTORS = [code for codes in INDUSTRY_GROUPS.values() for code in codes]
HIGHER_IS_BETTER = {
    "gdp_growth": True, "gdp_volatility": False,
    "unemployment_level": False, "unemployment_volatility": False,
    "industry_hhi": False, "establishment_growth": True,
    "population_growth": True, "net_migration_rate": True,
}
WEIGHTS = pd.Series({  # Original eight-component weights.
    "gdp_growth": 0.125,
    "gdp_volatility": 0.125,
    "unemployment_level": 0.125,
    "unemployment_volatility": 0.125,
    "industry_hhi": 0.125,
    "establishment_growth": 0.125,
    "population_growth": 0.125,
    "net_migration_rate": 0.125,
}, dtype=float)

if DEFINITION_MODE not in {"dataset_2024", "reference_periods"}:
    raise ValueError("DEFINITION_MODE must be 'dataset_2024' or 'reference_periods'.")
if list(WEIGHTS.index) != METRICS or not np.isfinite(WEIGHTS).all():
    raise ValueError("Provide one finite weight for each metric, in the displayed order.")
if (WEIGHTS <= 0).any() or not np.isclose(WEIGHTS.sum(), 1, rtol=0, atol=1e-9):
    raise ValueError("All eight weights must be positive and sum to 1.")
if not 0 <= SENSITIVITY < 1:
    raise ValueError("SENSITIVITY must be at least 0 and less than 1.")
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Parquet input not found: {DATA_PATH.resolve()}. Update DATA_PATH above.")
if not INDUSTRY_DATA_PATH.is_file():
    raise FileNotFoundError(f"Industry Parquet input not found: {INDUSTRY_DATA_PATH.resolve()}. Update INDUSTRY_DATA_PATH above.")
if len(CLASSIFIED_SECTORS) != 19 or len(set(CLASSIFIED_SECTORS)) != 19:
    raise ValueError("The industry grouping must contain 19 distinct classified NAICS codes.")

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 80)

## 2. Load and validate the panel

The data dictionary defines one metro-panel row per `cbsa_code` and `year` and one sector-panel row per `cbsa_code`, `year`, and `naics_code`. Five-digit metro and two-digit sector codes are preserved as strings. The analysis universe uses the metro names in the panel and excludes Puerto Rico (`", PR"`), consistent with the 50 states and District of Columbia in the reference framework. No county crosswalk or workbook is loaded. Duplicate keys cause an error rather than being silently combined.

In [3]:
panel = pd.read_parquet(DATA_PATH).copy()
required = {
    "cbsa_code", "cbsa_title", "year", "bea_gdp_usd", "bls_unemployment_rate_pct",
    "cbp_establishments_count", "cbp_employment_count", "pep_population_count", "pep_migration_net",
}
missing_columns = sorted(required - set(panel.columns))
if missing_columns:
    raise ValueError(f"Missing required Parquet columns: {missing_columns}")
if panel[["cbsa_code", "cbsa_title", "year"]].isna().any().any():
    raise ValueError("CBSA code, title, and year must be present on every row.")
panel["cbsa_code"] = panel["cbsa_code"].astype(str).str.strip().str.zfill(5)
if not panel["cbsa_code"].str.fullmatch(r"\d{5}").all():
    raise ValueError("Every CBSA code must have five numeric digits.")
years = pd.to_numeric(panel["year"], errors="raise")
if not np.isfinite(years).all() or not np.equal(years, np.floor(years)).all():
    raise ValueError("Every year must be a finite integer.")
panel["year"] = years.astype(int)
if panel.duplicated(["cbsa_code", "year"]).any():
    raise ValueError("More than one record exists for a CBSA and year.")
for column in required - {"cbsa_code", "cbsa_title", "year"}:
    panel[column] = pd.to_numeric(panel[column], errors="raise")

latest_titles = panel.sort_values("year").drop_duplicates("cbsa_code", keep="last")
puerto_rico = latest_titles["cbsa_title"].str.strip().str.endswith(", PR")
universe = (latest_titles.loc[~puerto_rico, ["cbsa_code", "cbsa_title"]]
            .set_index("cbsa_code").sort_index())
universe.index.name = "cbsa_code"
if len(universe) < 2:
    raise ValueError("The scoring universe must contain at least two metro areas.")
panel = panel.loc[panel["cbsa_code"].isin(universe.index)].copy()

industry_panel = pd.read_parquet(INDUSTRY_DATA_PATH).copy()
sector_required = {"cbsa_code", "year", "naics_code", "cbp_employment_count", "cbp_counties_reported_count"}
if sector_required - set(industry_panel.columns):
    raise ValueError(f"Missing required industry columns: {sorted(sector_required - set(industry_panel.columns))}")
if industry_panel[["cbsa_code", "year", "naics_code"]].isna().any().any():
    raise ValueError("Industry CBSA, year, and NAICS code must be present.")
industry_panel["cbsa_code"] = industry_panel["cbsa_code"].astype(str).str.strip().str.zfill(5)
industry_panel["naics_code"] = industry_panel["naics_code"].astype(str).str.strip().str.zfill(2)
if not industry_panel["cbsa_code"].str.fullmatch(r"\d{5}").all() or not industry_panel["naics_code"].str.fullmatch(r"\d{2}").all():
    raise ValueError("Industry CBSA codes must be five digits and NAICS sectors must be two digits.")
sector_years = pd.to_numeric(industry_panel["year"], errors="raise")
if not np.isfinite(sector_years).all() or not np.equal(sector_years, np.floor(sector_years)).all():
    raise ValueError("Every industry year must be a finite integer.")
industry_panel["year"] = sector_years.astype(int)
for column in ["cbp_employment_count", "cbp_counties_reported_count"]:
    industry_panel[column] = pd.to_numeric(industry_panel[column], errors="raise")
    if (industry_panel[column].dropna() < 0).any():
        raise ValueError(f"Negative industry value in {column}.")
if industry_panel.duplicated(["cbsa_code", "year", "naics_code"]).any():
    raise ValueError("More than one record exists for a CBSA, year, and NAICS sector.")
industry_panel = industry_panel.loc[industry_panel["cbsa_code"].isin(universe.index)].copy()

def sha256(path):
    hasher = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1_048_576), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

print(f"Metro input: {DATA_PATH.name} | SHA-256: {sha256(DATA_PATH)}")
print(f"Industry input: {INDUSTRY_DATA_PATH.name} | SHA-256: {sha256(INDUSTRY_DATA_PATH)}")
print(f"Years present: {panel.year.min()}–{panel.year.max()} | "
      f"U.S. MSAs: {len(universe)} | Puerto Rico MSAs excluded: {int(puerto_rico.sum())}")
print(f"Industry years present: {industry_panel.year.min()}–{industry_panel.year.max()} | Definition mode: {DEFINITION_MODE}")

Metro input: msa_year.parquet | SHA-256: b07364d18d9a2ffd640486b39c96f8d310cd3651d60a257cfccaa2b3b489dc10
Industry input: msa_naics_year.parquet | SHA-256: 210dd8dec2df91c8778ff78b50619cf767661cdb1347163e15b02c6e4d300e3d
Years present: 2015–2024 | U.S. MSAs: 387 | Puerto Rico MSAs excluded: 6
Industry years present: 2015–2023 | Definition mode: dataset_2024


## 3. Metric definitions

The reference framework uses annual nominal GDP, unemployment, private-supersector employment, employer establishments, population, and net migration. Each raw metric is rounded to 12 decimal places before percentile ranking. Unemployment is stored as a percent in this dataset and converted to a decimal; net migration is measured per 1,000 residents and may be negative.

**Important source distinctions:** The metro panel calculates the unemployment rate after summing county unemployment and labor-force counts. It is not the published BLS metro rate used in the reference workbook. Establishment growth uses all employer establishments, not just those with 20+ employees, and is not a measure of business births or net business formation. Industry HHI uses CBP 2023 sector employment, not QCEW 2024 annual-average employment. Bank-deposit HHI measures a different kind of concentration and is not used.

In [4]:
if DEFINITION_MODE == "dataset_2024":
    # Nine annual GDP growth rates need GDP for 2015–2024.
    gdp_growth_years = list(range(2016, 2025))
    population_start, population_end, migration_year = 2019, 2024, 2024
else:
    # The reference's ten growth rates need GDP for 2014–2024.
    gdp_growth_years = list(range(2015, 2025))
    population_start, population_end, migration_year = 2020, 2025, 2025

definitions = pd.DataFrame([
    ("GDP growth", "bea_gdp_usd", "2019–2024 CAGR", "Higher"),
    ("GDP volatility", "bea_gdp_usd", f"{gdp_growth_years[0]}–2024 annual growth; sample SD ({len(gdp_growth_years)} rates)", "Lower"),
    ("Unemployment level", "bls_unemployment_rate_pct", "2024 annual rate / 100", "Lower"),
    ("Unemployment volatility", "bls_unemployment_rate_pct", "2015–2024 annual rates / 100; sample SD (10 rates)", "Lower"),
    ("Industry HHI", "msa_naics_year: cbp_employment_count", f"{INDUSTRY_YEAR} CBP; 19 sectors grouped into 10 categories" if DEFINITION_MODE == "dataset_2024" else "2024 QCEW supersector data not supplied; unavailable", "Lower"),
    ("Establishment growth", "cbp_establishments_count", "2018–2023 CAGR, all employer establishments", "Higher"),
    ("Population growth", "pep_population_count", f"{population_start}–{population_end} CAGR", "Higher"),
    ("Net migration /1,000", "pep_migration_net; pep_population_count", f"{migration_year} net migration / average {migration_year - 1}–{migration_year} population × 1,000", "Higher"),
], columns=["metric", "panel column", "definition", "more attractive"])
display(definitions)
group_mapping = pd.DataFrame({"industry category": list(INDUSTRY_GROUPS),
                              "NAICS sector codes": [", ".join(v) for v in INDUSTRY_GROUPS.values()]})
display(group_mapping)

,metric,panel column,definition,more attractive
0,GDP growth,bea_gdp_usd,2019–2024 CAGR,Higher
1,GDP volatility,bea_gdp_usd,2016–2024 annual growth; sample SD (9 rates),Lower
2,Unemployment level,bls_unemployment_rate_pct,2024 annual rate / 100,Lower
3,Unemployment volatility,bls_unemployment_rate_pct,2015–2024 annual rates / 100; sample SD (10 rates),Lower
4,Industry HHI,msa_naics_year: cbp_employment_count,2023 CBP; 19 sectors grouped into 10 categories,Lower
5,Establishment growth,cbp_establishments_count,"2018–2023 CAGR, all employer establishments",Higher
6,Population growth,pep_population_count,2019–2024 CAGR,Higher
7,"Net migration /1,000",pep_migration_net; pep_population_count,"2024 net migration / average 2023–2024 population × 1,000",Higher


,industry category,NAICS sector codes
0,Natural resources and mining,"11, 21"
1,Construction,23
2,Manufacturing,31
3,"Trade, transportation, and utilities","22, 42, 44, 48"
4,Information,51
5,Financial activities,"52, 53"
6,Professional and business services,"54, 55, 56"
7,Education and health services,"61, 62"
8,Leisure and hospitality,"71, 72"
9,Other services,81


## 4. Calculate raw metrics from the Parquet data

Growth is calculated as `(ending / beginning) ** (1 / years) - 1`. Volatility is the sample standard deviation (`ddof=1`) and requires every observation in the selected window. Migration uses the aggregate metro flow divided by average beginning and ending metro population; it does not average county rates. No missing metric is filled with zero.

For industry concentration, missing sector rows are not treated as zero. A valid HHI requires all 19 classified sector employment values; `99` (unclassified) is excluded. In this dataset `31` covers 31–33, `44` covers 44–45, and `48` covers 48–49. A sector can still be understated when Census omits small county-sector observations. The diagnostic compares summed classified sector employment to the all-industry metro total, which includes unclassified employment, and shows the range of counties reported across sectors; these checks do not prove complete county coverage.

Population and migration counts may also be based on incomplete county coverage. The dataset includes the number of counties with reported population but not the independent expected county total, so this notebook can flag changes in reported counts across years; it cannot prove full coverage. The data dictionary identifies population through 2019 as intercensal estimates and population from 2020 onward as Vintage 2025, with both based on the 2020 Census. It also notes that pre-2020 Connecticut population and pre-2024 Connecticut GDP are unavailable because of changes in county-equivalent geography.

In [5]:
def panel_wide(column, requested_years):
    """Align requested years to the full U.S. metro universe; absent years stay missing."""
    requested_years = list(requested_years)
    return (panel.pivot(index="cbsa_code", columns="year", values=column)
            .reindex(index=universe.index, columns=requested_years).astype(float))

def cagr(start, end, years):
    valid = start.notna() & end.notna() & (start > 0) & (end >= 0)
    out = pd.Series(np.nan, index=start.index, dtype=float)
    out.loc[valid] = (end.loc[valid] / start.loc[valid]) ** (1 / years) - 1
    return out.round(12)

gdp_years = sorted(set([2019, 2024] + gdp_growth_years + [y - 1 for y in gdp_growth_years]))
gdp = panel_wide("bea_gdp_usd", gdp_years)
unemployment_years = list(range(2015, 2025))
unemployment = panel_wide("bls_unemployment_rate_pct", unemployment_years) / 100.0
cbp = panel_wide("cbp_establishments_count", [2018, 2023])
population_years = sorted({population_start, population_end, migration_year - 1, migration_year})
population = panel_wide("pep_population_count", population_years)
migration = panel_wide("pep_migration_net", [migration_year])

raw = pd.DataFrame(index=universe.index)
raw["gdp_growth"] = cagr(gdp[2019], gdp[2024], 5)
previous_gdp = gdp[[y - 1 for y in gdp_growth_years]].copy()
previous_gdp.columns = gdp_growth_years
current_gdp = gdp[gdp_growth_years]
annual_growth = (current_gdp / previous_gdp - 1).where(previous_gdp > 0)
raw["gdp_volatility"] = (annual_growth.std(axis=1, ddof=1)
                         .where(annual_growth.notna().sum(axis=1).eq(len(gdp_growth_years))).round(12))
raw["unemployment_level"] = unemployment[2024].round(12)
raw["unemployment_volatility"] = (unemployment.std(axis=1, ddof=1)
                                  .where(unemployment.notna().sum(axis=1).eq(10)).round(12))
# Group classified two-digit CBP NAICS sectors into ten reference-style categories.
# Any missing constituent sector makes its category—and thus HHI—unavailable.
industry_2023 = industry_panel.loc[industry_panel.year.eq(INDUSTRY_YEAR)]
industry_sectors = (industry_2023.pivot(index="cbsa_code", columns="naics_code", values="cbp_employment_count")
                    .reindex(index=universe.index, columns=CLASSIFIED_SECTORS).astype(float))
industry_counties = (industry_2023.pivot(index="cbsa_code", columns="naics_code", values="cbp_counties_reported_count")
                     .reindex(index=universe.index, columns=CLASSIFIED_SECTORS).astype(float))
industry_groups = pd.DataFrame({
    group: industry_sectors[codes].sum(axis=1, min_count=len(codes))
    for group, codes in INDUSTRY_GROUPS.items()
})
classified_total = industry_groups.sum(axis=1, min_count=len(INDUSTRY_GROUPS))
cbp_industry_hhi = (industry_groups.pow(2).sum(axis=1, min_count=len(INDUSTRY_GROUPS))
                    / classified_total.pow(2)).where(classified_total > 0).round(12)
raw["industry_hhi"] = cbp_industry_hhi if DEFINITION_MODE == "dataset_2024" else np.nan

total_cbp_jobs = panel_wide("cbp_employment_count", [INDUSTRY_YEAR])[INDUSTRY_YEAR]
industry_audit = pd.DataFrame({
    "published_classified_sectors": industry_sectors.notna().sum(axis=1),
    "missing_naics_sectors": industry_sectors.isna().apply(
        lambda row: ", ".join(code for code in CLASSIFIED_SECTORS if row[code]), axis=1),
    "classified_jobs": classified_total,
    "all_industry_jobs": total_cbp_jobs,
    "classified_to_all_industry_ratio": (classified_total / total_cbp_jobs).where(total_cbp_jobs > 0),
    "min_counties_reported": industry_counties.min(axis=1),
    "max_counties_reported": industry_counties.max(axis=1),
    "cbp_industry_hhi": cbp_industry_hhi,
}, index=universe.index)
raw["establishment_growth"] = cagr(cbp[2018], cbp[2023], 5)
raw["population_growth"] = cagr(population[population_start], population[population_end], population_end - population_start)
average_population = (population[migration_year - 1] + population[migration_year]) / 2
raw["net_migration_rate"] = (migration[migration_year] / average_population * 1_000).where(average_population > 0).round(12)
raw = raw[METRICS].replace([np.inf, -np.inf], np.nan)

coverage = pd.DataFrame({
    "metric": [METRIC_LABELS[m] for m in METRICS],
    "available_metros": raw.notna().sum().values,
    "missing_metros": raw.isna().sum().values,
})
display(coverage)
print(f"Industry: {int(cbp_industry_hhi.notna().sum())} metros have all 19 classified sector values "
      f"for CBP {INDUSTRY_YEAR}. Sector presence does not prove full county coverage.")

if "pep_counties_reported_count" in panel.columns:
    county_counts = panel_wide("pep_counties_reported_count", population_years)
    counts_known = county_counts.notna().all(axis=1)
    inconsistent = counts_known & county_counts.nunique(axis=1, dropna=True).gt(1)
    print(f"Population reporting counts differ across the required years in {int(inconsistent.sum())} of "
          f"{int(counts_known.sum())} metros with counts present for every year; "
          "this check cannot establish full coverage.")
    if inconsistent.any():
        display(universe.loc[inconsistent].join(county_counts.loc[inconsistent]).head(20))
else:
    print("Population county-reporting count is absent; county coverage could not be checked.")

,metric,available_metros,missing_metros
0,GDP growth,382,5
1,GDP volatility,382,5
2,Unemployment level,387,0
3,Unemployment volatility,387,0
4,Industry HHI,292,95
5,Establishment growth,382,5
6,Population growth,382,5
7,"Net migration /1,000",387,0


Industry: 292 metros have all 19 classified sector values for CBP 2023. Sector presence does not prove full county coverage.
Population reporting counts differ across the required years in 0 of 382 metros with counts present for every year; this check cannot establish full coverage.


## 5. Convert raw metrics to oriented percentiles

For each metric, rank only the markets with a valid observation. Higher growth and migration are better; lower volatility, unemployment, and industry concentration are better. Orient values so the least attractive gets 0 and the most attractive gets 100:

`percentile = 100 × (average oriented rank − 1) / (valid count − 1)`

Ties receive the average rank. With fewer than two valid observations there is no percentile. Scores are calculated before any display filtering, so selecting Charlotte does not change the comparison cohort.

In [6]:
def oriented_percentiles(values, higher_is_better):
    values = pd.to_numeric(values, errors="raise")
    n = int(values.notna().sum())
    if n < 2:
        return pd.Series(np.nan, index=values.index, dtype=float)
    oriented = values if higher_is_better else -values
    return 100.0 * (oriented.rank(method="average", ascending=True) - 1) / (n - 1)

percentiles = pd.DataFrame({m: oriented_percentiles(raw[m], HIGHER_IS_BETTER[m]) for m in METRICS})

## 6. Scores, provisional ranks, and missing-data bounds

The **available-data score** divides observed weighted percentile points by observed weight. It can be calculated for a metro with fewer than eight metrics, but should not be ranked alongside metros with a different set of metrics.

The **complete eight-metric score** and rank require all eight measures under the selected definition mode. In the default mode these are adapted scores using 2023 CBP for industry and the 2024-panel windows, not exact workbook scores. The secondary **seven-metric score excluding industry** requires all seven other metrics and ranks metros against the same seven-metric cohort; it is a separate sensitivity view and should not be mixed with the eight-metric ranking.

The all-market lower and upper bounds reproduce the reference's missing-observation rank bounds for the eight-metric framework. A market's own missing component spans 0–100; for observed components, other markets' missing values can fall on either side. They are conservative sensitivity ranges, not confidence intervals.

In [7]:
def weighted_score(percentile_frame, weights):
    """Return observed score, observed original weight, and valid metric count."""
    weights = weights.reindex(percentile_frame.columns)
    observed = percentile_frame.notna()
    available_weight = observed.mul(weights, axis=1).sum(axis=1)
    points = percentile_frame.mul(weights, axis=1).sum(axis=1, min_count=1)
    score = points.div(available_weight.where(available_weight > 0))
    return score, available_weight, observed.sum(axis=1)

available_score, weight_coverage, valid_metrics = weighted_score(percentiles, WEIGHTS)
complete_score = available_score.where(valid_metrics.eq(len(METRICS)))
complete_rank = complete_score.rank(method="min", ascending=False)
interim_weights = WEIGHTS[INTERIM_METRICS] / WEIGHTS[INTERIM_METRICS].sum()
interim_available, _, interim_count = weighted_score(percentiles[INTERIM_METRICS], interim_weights)
interim_score = interim_available.where(interim_count.eq(len(INTERIM_METRICS)))
interim_rank = interim_score.rank(method="min", ascending=False)

n_universe = len(universe)
lower = pd.DataFrame(index=universe.index, columns=METRICS, dtype=float)
upper = lower.copy()
for metric in METRICS:
    n_valid = int(raw[metric].notna().sum())
    pct = percentiles[metric]
    lower[metric] = (pct * (n_valid - 1) / (n_universe - 1)).fillna(0)
    upper[metric] = (lower[metric] + 100 * (n_universe - n_valid) / (n_universe - 1)).where(pct.notna(), 100)

scores = pd.DataFrame({
    "complete_8_score": complete_score,
    "complete_8_rank": complete_rank,
    "excluding_industry_7_score": interim_score,
    "excluding_industry_7_rank": interim_rank,
    "available_data_score": available_score,
    "valid_metrics_out_of_8": valid_metrics,
    "original_weight_coverage": weight_coverage,
    "all_market_lower_8_metric": lower.mul(WEIGHTS, axis=1).sum(axis=1),
    "all_market_upper_8_metric": upper.mul(WEIGHTS, axis=1).sum(axis=1),
}, index=universe.index)
scores["coverage_status"] = np.where(
    scores.complete_8_score.notna(), "Complete (8/8)",
    np.where(scores.excluding_industry_7_score.notna(), "Partial (7/8; industry unavailable)", "Partial; no comparable rank")
)
scores["missing_metrics"] = raw.isna().apply(
    lambda row: "; ".join(METRIC_LABELS[m] for m in METRICS if row[m]), axis=1
)

results = (universe.join(scores).join(raw.add_suffix("_raw"))
           .join(percentiles.add_suffix("_percentile"))
           .join(industry_audit[["published_classified_sectors", "missing_naics_sectors", "classified_to_all_industry_ratio"]]))
results = results.sort_values(["complete_8_rank", "cbsa_title"], na_position="last")
print(f"Complete eight-metric scores: {int(complete_score.notna().sum())}")
print(f"Partial eight-metric records: {int(complete_score.isna().sum())}")
print(f"Comparable seven-metric scores, excluding industry: {int(interim_score.notna().sum())}")
display(results[["cbsa_title", "complete_8_score", "complete_8_rank", "original_weight_coverage",
                 "all_market_lower_8_metric", "all_market_upper_8_metric", "missing_metrics"]].head(20))

Complete eight-metric scores: 287
Partial eight-metric records: 100
Comparable seven-metric scores, excluding industry: 382


,cbsa_title,complete_8_score,complete_8_rank,original_weight_coverage,all_market_lower_8_metric,all_market_upper_8_metric,missing_metrics
cbsa_code,,,,,,,
39340,"Provo-Orem-Lehi, UT",92.276709,1.0,1.0,88.860104,92.584197,
14260,"Boise City, ID",84.918141,2.0,1.0,81.573834,85.297927,
41100,"St. George, UT",84.880024,3.0,1.0,83.031088,86.755181,
39580,"Raleigh-Cary, NC",84.233457,4.0,1.0,81.282383,85.006477,
22220,"Fayetteville-Springdale-Rogers, AR",83.800232,5.0,1.0,82.091969,85.816062,
12420,"Austin-Round Rock-San Marcos, TX",82.365756,6.0,1.0,78.853627,82.577720,
38060,"Phoenix-Mesa-Chandler, AZ",82.188377,7.0,1.0,78.918394,82.642487,
43620,"Sioux Falls, SD-MN",81.859073,8.0,1.0,79.047927,82.772021,
14580,"Bozeman, MT",81.367306,9.0,1.0,77.979275,81.703368,


## 7. Weight sensitivity for the eight-metric ranking

Each metric's weight is increased and decreased by 25%, one at a time, while all eight weights are renormalized. Scores and competition ranks are calculated only for the complete-eight-metric cohort. `best_weight_rank` and `worst_weight_rank` span the 16 individual scenarios **and** the baseline; they are not exhaustive simultaneous-weight bounds.

In [8]:
scenario_scores = pd.DataFrame(index=universe.index)
scenario_ranks = pd.DataFrame(index=universe.index)
for metric in METRICS:
    for direction, sign in [("down", -1), ("up", 1)]:
        modified = WEIGHTS.copy()
        modified[metric] *= 1 + sign * SENSITIVITY
        modified /= modified.sum()
        candidate, _, candidate_n = weighted_score(percentiles[METRICS], modified)
        scenario = candidate.where(candidate_n.eq(len(METRICS)))
        key = f"{metric}_{direction}"
        scenario_scores[key] = scenario
        scenario_ranks[key] = scenario.rank(method="min", ascending=False)

all_complete_ranks = pd.concat([complete_rank.rename("baseline"), scenario_ranks], axis=1)
results["best_weight_rank"] = all_complete_ranks.min(axis=1)
results["worst_weight_rank"] = all_complete_ranks.max(axis=1)
print(f"Scenarios: {scenario_ranks.shape[1]}, plus the baseline.")

Scenarios: 16, plus the baseline.


## 8. Inspect a market

The contribution shown below uses the eight-metric weights. For a complete market, the contributions sum to its score. For a partial market, divide observed contributions by `original_weight_coverage` to recover the available-data score; its score is not given a complete-market rank. The industry audit makes its sector coverage visible.

In [9]:
if FOCUS_CBSA in universe.index:
    focus = pd.DataFrame({
        "metric": [METRIC_LABELS[m] for m in METRICS],
        "raw_value": raw.loc[FOCUS_CBSA, METRICS].to_numpy(),
        "percentile": percentiles.loc[FOCUS_CBSA, METRICS].to_numpy(),
        "original_weight": WEIGHTS.loc[METRICS].to_numpy(),
    })
    focus["original_weighted_points"] = focus["percentile"] * focus["original_weight"]
    print(f"{FOCUS_CBSA}: {universe.loc[FOCUS_CBSA, 'cbsa_title']}")
    display(focus)
    display(results.loc[[FOCUS_CBSA], ["complete_8_score", "complete_8_rank", "excluding_industry_7_score", "excluding_industry_7_rank",
            "valid_metrics_out_of_8", "original_weight_coverage", "all_market_lower_8_metric",
            "all_market_upper_8_metric", "best_weight_rank", "worst_weight_rank", "missing_metrics"]])
    print(f"CBP {INDUSTRY_YEAR} industry-sector audit:")
    display(industry_audit.loc[[FOCUS_CBSA]])
    display(industry_groups.loc[[FOCUS_CBSA]].T.rename(columns={FOCUS_CBSA: "employment"}))
else:
    print(f"CBSA {FOCUS_CBSA} is outside the scoring universe.")

16740: Charlotte-Concord-Gastonia, NC-SC


,metric,raw_value,percentile,original_weight,original_weighted_points
0,GDP growth,0.083104,87.926509,0.125,10.990814
1,GDP volatility,0.031938,75.328084,0.125,9.416010
2,Unemployment level,0.036902,55.958549,0.125,6.994819
3,Unemployment volatility,0.011782,58.808290,0.125,7.351036
4,Industry HHI,0.148470,88.316151,0.125,11.039519
5,Establishment growth,0.025970,88.713911,0.125,11.089239
6,Population growth,0.018898,90.551181,0.125,11.318898
7,"Net migration /1,000",19.627987,92.487047,0.125,11.560881


,complete_8_score,complete_8_rank,excluding_industry_7_score,excluding_industry_7_rank,valid_metrics_out_of_8,original_weight_coverage,all_market_lower_8_metric,all_market_upper_8_metric,best_weight_rank,worst_weight_rank,missing_metrics
cbsa_code,,,,,,,,,,,
16740,79.761215,13.0,78.539082,22.0,8,1.0,76.489637,80.213731,11.0,14.0,


CBP 2023 industry-sector audit:


,published_classified_sectors,missing_naics_sectors,classified_jobs,all_industry_jobs,classified_to_all_industry_ratio,min_counties_reported,max_counties_reported,cbp_industry_hhi
cbsa_code,,,,,,,,
16740,19,,1237866.0,1238884.0,0.999178,4.0,11.0,0.14847


cbsa_code,employment
Natural resources and mining,1098.0
Construction,73922.0
Manufacturing,110240.0
"Trade, transportation, and utilities",299241.0
Information,29354.0
Financial activities,131214.0
Professional and business services,216169.0
Education and health services,172465.0
Leisure and hospitality,151805.0
Other services,52358.0


## 9. Checks and optional export

`results` holds one row per metro, with scores, ranks, coverage, raw metrics, percentiles, and selected sector diagnostics. `industry_audit` and `industry_groups` hold detailed CBP coverage and grouped employment, while `scenario_scores` and `scenario_ranks` contain the 16 individual weight scenarios. Set `EXPORT_CSV = True` above to write the results table.

In [ ]:
assert results.index.is_unique
assert complete_score.notna().equals(valid_metrics.eq(len(METRICS)))
assert interim_score.notna().equals(interim_count.eq(len(INTERIM_METRICS)))
assert ((percentiles >= 0) & (percentiles <= 100) | percentiles.isna()).all().all()
assert (scores.all_market_lower_8_metric <= scores.all_market_upper_8_metric + 1e-10).all()
assert results.loc[results.complete_8_rank.notna(), "best_weight_rank"].le(
    results.loc[results.complete_8_rank.notna(), "complete_8_rank"]).all()
assert results.loc[results.complete_8_rank.notna(), "worst_weight_rank"].ge(
    results.loc[results.complete_8_rank.notna(), "complete_8_rank"]).all()
print("Internal score and rank checks passed.")
if EXPORT_CSV:
    results.to_csv(OUTPUT_PATH, index=True, index_label="cbsa_code")
    print(f"Wrote {len(results):,} metro rows to {OUTPUT_PATH.resolve()}")
else:
    print("CSV export is off; set EXPORT_CSV = True to write the results table.")